# ALTO/VTRN에서 R1과 R1+R4 비교
Kaggle에서 GPU와 Internet을 켜고 Run All을 실행하세요. 공식 Round 2 Val.zip을 Add Input으로 연결하거나 자동 다운로드를 사용합니다. 같은 4방향, LoMa-B, 걸음 내 지도 특징 재사용, 지도 및 모의 오도메트리로 비교합니다.

R1은 중심 블록 매칭 실패 시 추측 위치를 유지합니다. R1+R4는 이웃 블록을 추가 탐색합니다. 약 800m 구간별 재생과 약 4.57km 연속 재생의 전체 시간, 걸음별 오차, 구간 최종 오차, 복구 횟수를 기록합니다. 양의 시간 차이는 R4 추가 소요 시간, 음의 오차 차이는 R4의 오차 감소입니다. 짧은 구간에서 800m 미만의 마지막 잔여 구간은 제외하며 연속 재생에는 포함합니다.

In [ ]:
from pathlib import Path
import io, json, sys, subprocess, shutil, zipfile, datetime, time
import pandas as pd
import numpy as np
import cv2, torch, requests
from IPython.display import FileLink, display
assert torch.cuda.is_available(), 'Enable Kaggle GPU'
WORK=Path('/kaggle/working/alto_r1_vs_r4')
RUN=WORK/datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
SOURCE=RUN/'source'
SOURCE.mkdir(parents=True)
DATA=WORK/'vtrn_val'
DATA.mkdir(exist_ok=True)
IMAGES=DATA/'images'
IMAGES.mkdir(exist_ok=True)
MAX_METRES=5000
SAMPLING_METRES=50
DROPBOX='https://www.dropbox.com/scl/fo/saejbf9qanbfq40k8jo18/h?dl=1&rlkey=l19kx1vzzahifv3n5lkg19n84'
def save_json(p,v): p.write_text(json.dumps(v,indent=2,default=str))

In [ ]:
# Select the continuous prefix of the official validation sequence
from pathlib import PurePosixPath
def select_frames(query):
    q=query.copy()
    q['frame']=q.name.map(lambda s:int(Path(s).stem))
    q=q.sort_values('frame').reset_index(drop=True)
    assert q.frame.is_unique and np.isfinite(q[['easting','northing']]).all().all()
    d=np.linalg.norm(np.diff(q[['easting','northing']],axis=0),axis=1)
    cuts=np.flatnonzero((np.diff(q.frame)!=1)|(d>50))
    if len(cuts): q=q.iloc[:cuts[0]+1].copy()
    q['distance']=np.r_[0,np.cumsum(d[:len(q)-1])]
    picked=[0]
    for i,dist in enumerate(q.distance):
        if dist>MAX_METRES: break
        if dist-q.distance.iloc[picked[-1]]>=SAMPLING_METRES: picked.append(i)
    result=q.iloc[picked].copy()
    assert result.distance.iloc[-1]>=600, 'Continuous validation prefix too short'
    return result
def wanted():
    return select_frames(pd.read_csv(DATA/'query.csv')).name.tolist()
mount=Path('/kaggle/input')
loose=[p.parent for p in mount.rglob('query.csv') if (p.parent/'query_images').is_dir() and (p.parent/'reference.csv').exists()]
bundles=list(mount.rglob('Val.zip'))
if loose:
    parent=loose[0]
    for name in ('query.csv','reference.csv'): shutil.copyfile(parent/name,DATA/name)
    for name in wanted(): shutil.copyfile(parent/'query_images'/name,IMAGES/Path(name).name)
elif bundles:
    with zipfile.ZipFile(bundles[0]) as z:
        prefix=next(x[:-9] for x in z.namelist() if x.endswith('/query.csv') or x=='query.csv')
        for name in ('query.csv','reference.csv'): (DATA/name).write_bytes(z.read(prefix+name))
        for name in wanted(): (IMAGES/Path(name).name).write_bytes(z.read(prefix+'query_images/'+name))
else:
    subprocess.check_call([sys.executable,'-m','pip','-q','install','stream-unzip'])
    from stream_unzip import stream_unzip
    def consume(entries):
        needed=None
        for path,size,chunks in entries:
            p=PurePosixPath(path.decode())
            if p.name in ('query.csv','reference.csv'):
                (DATA/p.name).write_bytes(b''.join(chunks))
                if p.name=='query.csv': needed=set(wanted())
            elif needed is not None and p.parent.name=='query_images' and p.name in needed:
                (IMAGES/p.name).write_bytes(b''.join(chunks)); needed.remove(p.name)
            else:
                for _ in chunks: pass
            if needed==set() and (DATA/'reference.csv').exists(): return
        raise RuntimeError('Missing validation frames or metadata')
    with requests.get(DROPBOX,stream=True,timeout=(30,180)) as resp:
        resp.raise_for_status()
        # Dropbox streams an outer ZIP with an unsized, stored Val.zip member.
        # Start the inner streaming reader at the verified local-file boundary.
        import struct
        raw=resp.raw
        while True:
            header=raw.read(30)
            if len(header)!=30 or header[:4]!=b'PK\x03\x04':
                raise RuntimeError('Unexpected Round 2 bundle. Attach official Val.zip.')
            _,_,flags,method,_,_,_,compressed,_,nl,xl=struct.unpack('<I5H3I2H',header)
            name=raw.read(nl).decode()
            raw.read(xl)
            if name.rstrip('/').split('/')[-1]=='Val.zip' and method==0:
                consume(stream_unzip(iter(lambda:raw.read(1024*1024),b'')))
                break
            if not name.endswith('/') or compressed:
                raise RuntimeError('Unexpected archive order. Attach official Val.zip.')
frames=select_frames(pd.read_csv(DATA/'query.csv'))
assert all((IMAGES/Path(x).name).is_file() for x in frames.name)
frames.to_csv(RUN/'frames.csv',index=False)
print(len(frames),'frames;',round(frames.distance.iloc[-1]),'m')

In [ ]:
# Fixed north-up reference grid; map bounds from reference metadata and known start
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
session=requests.Session()
session.mount('https://',HTTPAdapter(max_retries=Retry(total=4,backoff_factor=1,status_forcelist=[429,500,502,503,504])))
maps=WORK/'naip_32617_1m'
maps.mkdir(exist_ok=True)
ref=pd.read_csv(DATA/'reference.csv')
xy=ref[['easting','northing']].to_numpy()
origin=frames[['easting','northing']].iloc[0].to_numpy(float)
xy=xy[np.linalg.norm(xy-origin,axis=1)<=MAX_METRES+768]
assert len(xy)>0
grid=np.unique(np.rint(xy/256).astype(int),axis=0)
tiles=sorted({(int(x+dx),int(y+dy)) for x,y in grid for dx in (-1,0,1) for dy in (-1,0,1)})
endpoint='https://imagery.nationalmap.gov/arcgis/rest/services/USGSNAIPImagery/ImageServer/exportImage'
for x,y in tiles:
    p=maps/f'{x}_{y}.jpg'
    if p.exists() and (maps/f'{x}_{y}.json').exists(): continue
    cx,cy=x*256,y*256
    bbox=[cx-256,cy-256,cx+256,cy+256]
    params=dict(bbox=','.join(map(str,bbox)),bboxSR=32617,imageSR=32617,size='512,512',format='jpg',f='json')
    response=session.get(endpoint,params=params,timeout=(30,120)); response.raise_for_status()
    info=response.json()
    if 'extent' not in info: raise RuntimeError(info)
    ext=info['extent']
    assert np.allclose([ext[k] for k in ('xmin','ymin','xmax','ymax')],bbox,rtol=0,atol=.01)
    response=session.get(info['href'],timeout=(30,120)); response.raise_for_status()
    im=cv2.imdecode(np.frombuffer(response.content,np.uint8),cv2.IMREAD_COLOR)
    assert im is not None and im.shape==(512,512,3)
    p.write_bytes(response.content);save_json(maps/f'{x}_{y}.json',info)
save_json(RUN/'grid.json',tiles)
print(len(tiles),'tiles')

In [ ]:
def make_env(name, packages):
    import sys
    import importlib.metadata as metadata
    env=WORK/'envs'/name
    py=env/'bin/python'
    env.parent.mkdir(parents=True,exist_ok=True)
    log=RUN/(name+'_environment.log')

    def command(args, required=True):
        result=subprocess.run([str(x) for x in args],stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT,text=True)
        output=result.stdout or ''
        with log.open('a',encoding='utf-8') as f:
            f.write(repr([str(x) for x in args])+chr(10)+output+chr(10))
        if output: print(output,flush=True)
        if required and result.returncode:
            raise RuntimeError(f'Environment command failed ({result.returncode}). See {log}: '+output[-4000:])
        return result

    def valid_python():
        if not py.exists(): return False
        try:
            probe=command([py,'-c',
                'import sys; from pathlib import Path; '
                'assert Path(sys.prefix).resolve()==Path(sys.argv[1]).resolve(); '
                'assert sys.prefix != sys.base_prefix',env],required=False)
            return probe.returncode==0
        except OSError:
            return False

    if not valid_python():
        # Kaggle may omit ensurepip. Create the interpreter without invoking it.
        created=command([sys.executable,'-m','venv','--without-pip',
                         '--system-site-packages',env],required=False)
        if created.returncode or not valid_python():
            # Fallback also handles installations without the stdlib venv module.
            bootstrap=WORK/'virtualenv_bootstrap'
            command([sys.executable,'-m','pip','install','--target',bootstrap,'virtualenv'])
            launcher='import sys,runpy; sys.path.insert(0,sys.argv.pop(1)); runpy.run_module("virtualenv",run_name="__main__")'
            command([sys.executable,'-c',launcher,bootstrap,'--system-site-packages',env])
        if not valid_python():
            raise RuntimeError(f'Invalid environment interpreter: {py}. See {log}')
    # Parent pip can install into a pip-less environment; no ensurepip needed.
    pip_check=command([py,'-m','pip','--version'],required=False)
    if pip_check.returncode:
        command([sys.executable,'-m','pip','--python',py,'install','pip'])
    command([py,'-m','pip','--version'])
    pins=[package+'=='+metadata.version(package) for package in ('torch','torchvision','numpy')]
    constraint=env/'constraints.txt'
    constraint.write_text(chr(10).join(pins),encoding='utf-8')
    command([py,'-m','pip','install','-c',constraint,*packages])
    frozen=command([py,'-m','pip','freeze'])
    (RUN/(name+'_pip-freeze.txt')).write_text(frozen.stdout,encoding='utf-8')
    return py


PY=make_env('loma',['vismatch==1.3.1','scipy'])

In [ ]:
sources={'imatch/geometry.py': '"""파이프라인 ④⑤: 대응점 검증(MAGSAC++)과 호모그래피 → 위치·heading.\n\n좌표 규칙 (Bearing-UAV와 같음)\n- 드론 영상 256 px, 지도 블록 512 px (2x2 타일). 블록 중심이 (0, 0), 1단위 = 128 px = 32 m.\n- 정답 변환: block_px = C + R(theta) (uav_px - 128),  R(t) = [[cos t, sin t], [-sin t, cos t]]\n"""\nimport numpy as np\n\nBLOCK_CENTER = 256.0\nPX_PER_UNIT = 128.0\nM_PER_UNIT = 32.0\n\nRANSAC_PX = 3.0  # MAGSAC++ 임계값(px). scripts/run.py --ransac-px 로 바꾼다\n\n\ndef set_ransac_px(px):\n    global RANSAC_PX\n    RANSAC_PX = float(px)\n\n\ndef rot(a):\n    t = np.deg2rad(a); c, s = np.cos(t), np.sin(t)\n    return np.array([[c, s], [-s, c]])\n\n\ndef turn(img, a):\n    """영상을 a도 돌린다 (반시계, cv2 규칙). 빈 모서리는 검은색."""\n    import cv2\n    h, w = img.shape[:2]\n    return cv2.warpAffine(img, cv2.getRotationMatrix2D((w / 2, h / 2), a, 1), (w, h), flags=cv2.INTER_LINEAR,\n                          borderMode=cv2.BORDER_CONSTANT, borderValue=0)\n\n\ndef wrap(d):\n    """각도 차이의 절댓값 (0~180도)."""\n    return np.abs((np.asarray(d) + 180) % 360 - 180)\n\n\ndef magsac(p0, p1):\n    """대응점 → (p0, p1, 호모그래피 H, inlier 마스크). 점이 4개 미만이면 H = None."""\n    import cv2\n    if len(p0) < 4:\n        return p0, p1, None, np.zeros(len(p0), bool)\n    H, mask = cv2.findHomography(p0, p1, cv2.USAC_MAGSAC, RANSAC_PX, maxIters=2000, confidence=0.995)\n    return p0, p1, H, (mask[:, 0].astype(bool) if mask is not None else np.zeros(len(p0), bool))\n\n\ndef pose_from_homography(H, scale, angle, uav_center):\n    """돌린 드론 영상 → 블록 호모그래피에서 위치(x, y: 블록 단위)와 heading(도), 축척을 구한다.\n\n    위치: 드론 영상 중심을 블록으로 옮긴 점. heading: 중심에서의 국소 선형 변환 L의 회전 성분.\n    회전 탐색으로 영상을 angle만큼 돌렸으므로 L에 R(angle)을 곱해 원래 영상 기준으로 되돌린다.\n    """\n    S = np.diag([scale, scale, 1.])\n    Hb = np.linalg.inv(S) @ H @ S                                   # 축척 보정 (base px 기준)\n    h = Hb @ np.array([uav_center[0], uav_center[1], 1.]); c = h[:2] / h[2]  # 회전해도 중심은 그대로\n    L = (Hb[:2, :2] - np.outer(c, Hb[2, :2])) / h[2] @ rot(angle)   # 중심에서의 야코비안\n    return dict(px=float((c[0] - BLOCK_CENTER) / PX_PER_UNIT), py=float((c[1] - BLOCK_CENTER) / PX_PER_UNIT),\n                head=float(np.degrees(np.arctan2(L[0, 1], L[0, 0]))), scale_est=float(np.sqrt(abs(np.linalg.det(L)))))\n', 'imatch/matchers.py': '"""파이프라인 ②③: 특징 추출과 매칭. 모든 매처는 (드론 영상, 블록) → magsac() 결과를 돌려준다.\n\n새 매처를 실험하려면 __call__(im0, im1) 을 가진 클래스를 만들고 get_matcher()에 이름을 추가하면 된다.\n"""\nimport os\n\nimport cv2\nimport numpy as np\n\nfrom .geometry import magsac\n\n# vismatch의 get_matcher에 없는 LoMa 변형은 직접 만든다 (\'loma\' = LoMa-B는 get_matcher로)\nLOMA_ARCH = {\'loma-b128\': \'LoMa-B128\', \'loma-l\': \'LoMa-L\', \'loma-g\': \'LoMa-G\'}\n\nclass VisMatch:\n    """vismatch 라이브러리의 공개 매처 (loma, romav2, sift-lightglue 등). GPU 권장.\n\n    LoMa 변형: \'loma\'(= LoMa-B, 주력), \'loma-b128\'(경량), \'loma-l\', \'loma-g\', \'loma-r\'.\n    """\n\n    def __init__(self, name, device=\'cuda\'):\n        import torch\n        self.name, self.torch = name, torch\n        if name in LOMA_ARCH:\n            from vismatch.im_models import loma\n            self.m = loma.LoMaMatcher(device, 2048, arch=LOMA_ARCH[name])\n        else:\n            from vismatch import get_matcher\n            self.m = get_matcher(name, device=device)\n        if name.startswith(\'loma\') and os.environ.get(\'LOMA_FP32\'):\n            # GTX 1650은 fp16에서 키포인트가 전부 NaN이 되어 오류 없이 짝 0개가 나온다. LOMA_FP32=1로 fp32를 쓴다.\n            # T4 결과(fp16)와 조금 다를 수 있으니 보고용 숫자는 T4 기본 설정으로 낸다\n            from vismatch.utils import force_float32\n            force_float32(self.m.matcher)\n        if name == \'romav2\':\n            # 기본 \'precise\' 설정은 T4(bf16 없음)에서 메모리 부족으로 멈췄다. 우리 영상은 256 / 512 px라\n            # \'base\' 설정(640 px, 단방향, fp32)을 쓴다.\n            from vismatch.im_models import romav2 as rv2\n            rv2._disable_autocast()\n            self.m.romav2_model.apply_setting(os.environ.get(\'ROMAV2_SETTING\', \'base\'))\n            self.name = f"romav2-{os.environ.get(\'ROMAV2_SETTING\', \'base\')}"\n\n    def __call__(self, im0, im1):\n        t = lambda im: self.torch.from_numpy(im).permute(2, 0, 1).float() / 255.\n        r = self.m(t(im0), t(im1))\n        p0, p1 = r[\'matched_kpts0\'], r[\'matched_kpts1\']\n        # vismatch는 inlier 점만 주고 마스크는 안 주므로 같은 점으로 MAGSAC++를 다시 돌린다\n        return magsac(np.float32(p0).reshape(-1, 2), np.float32(p1).reshape(-1, 2))\n\n\nclass CvSift:\n    """GPU 없이 도는 기준선: OpenCV SIFT + Lowe 비율 검사 + MAGSAC++. 설치·동작 확인용."""\n    name = \'cv-sift\'\n\n    def __init__(self):\n        self.sift = cv2.SIFT_create(4096); self.bf = cv2.BFMatcher()\n\n    def __call__(self, im0, im1):\n        k0, d0 = self.sift.detectAndCompute(cv2.cvtColor(im0, cv2.COLOR_RGB2GRAY), None)\n        k1, d1 = self.sift.detectAndCompute(cv2.cvtColor(im1, cv2.COLOR_RGB2GRAY), None)\n        if d0 is None or d1 is None or len(k0) < 2 or len(k1) < 2:\n            return np.empty((0, 2)), np.empty((0, 2)), None, np.zeros(0, bool)\n        mm = [m for m, n in self.bf.knnMatch(d0, d1, k=2) if m.distance < 0.8 * n.distance]\n        p0 = np.float32([k0[m.queryIdx].pt for m in mm]).reshape(-1, 2)\n        p1 = np.float32([k1[m.trainIdx].pt for m in mm]).reshape(-1, 2)\n        return magsac(p0, p1)\n\n\nclass VggMNN:\n    """비교 실험 \'VGG16 백본은 그대로 두면?\' (docs/02_실험기록.md 7장). 원 Bearing-UAV 체크포인트의 고정\n    VGG16 conv4_3(B2가 상관을 계산하는 그 특징)을 점 단위 기술자로 쓴다.\n\n    특징 맵의 칸(stride 8, 중심 8 i + 4)이 점이다. 블록 평균을 뺀 코사인 유사도로 상호 최근접 짝을 짓고,\n    이후는 다른 매처와 같은 MAGSAC++ / 호모그래피를 쓴다. 회전으로 생긴 검은 모서리의 칸은 뺀다.\n    capstone1 저장소와 체크포인트가 필요하다:\n      CAPSTONE1_DIR=<capstone1 경로> VGG_CKPT=Bearing_UAV/cross_view python scripts/run.py --matchers vgg-mnn --device cpu\n    결과: Singapore 44%(B2 69%, 100샘플) → 원 모델 특징으로는 점 단위 매칭이 안 된다.\n    """\n    name = \'vgg-mnn\'\n\n    def __init__(self):\n        import sys\n        import torch\n        root = os.environ[\'CAPSTONE1_DIR\']\n        sys.path.insert(0, root)\n        from diag.diag_modules import DEV, load_model, to_tensor\n        self.torch, self.tt, self.dev = torch, to_tensor, DEV\n        self.bb = load_model(os.path.join(root, os.environ.get(\'VGG_CKPT\', \'Bearing_UAV/cross_view\'))).backbone\n\n    def _feat(self, im):\n        with self.torch.no_grad():\n            f = self.bb(self.tt(im).unsqueeze(0).to(self.dev))[0]          # [C, h, w]\n        C, h, w = f.shape; st = im.shape[0] / h\n        yy, xx = np.mgrid[0:h, 0:w]\n        pts = np.stack([xx.ravel() * st + st / 2, yy.ravel() * st + st / 2], 1).astype(np.float32)\n        return f.reshape(C, -1).T, pts\n\n    def __call__(self, im0, im1):\n        F = self.torch.nn.functional\n        d0, p0 = self._feat(im0); d1, p1 = self._feat(im1)\n        valid = im0[np.clip(p0[:, 1].astype(int), 0, im0.shape[0] - 1), np.clip(p0[:, 0].astype(int), 0, im0.shape[1] - 1)].sum(1) > 0\n        d0, p0 = d0[self.torch.from_numpy(valid)], p0[valid]\n        mu = d1.mean(0, keepdim=True)\n        d0, d1 = F.normalize(d0 - mu, dim=1), F.normalize(d1 - mu, dim=1)\n        S = d0 @ d1.T\n        j = S.argmax(1); i = S.argmax(0)\n        mutual = (i[j] == self.torch.arange(len(j), device=j.device)).cpu().numpy()\n        j = j.cpu().numpy()\n        return magsac(p0[mutual], p1[j[mutual]])\n\n\ndef get_matcher(name, device=\'cuda\'):\n    if name == \'cv-sift\':\n        return CvSift()\n    if name == \'vgg-mnn\':\n        return VggMNN()\n    return VisMatch(name, device)\n', 'imatch/pipeline.py': '"""파이프라인 ①~⑤를 한 샘플에 적용한다.\n\n① 회전 탐색: 드론 영상을 angles의 각 각도로 돌린다 (angles=[0]이면 탐색 없음, 원 Bearing-UAV와 같은 조건)\n②③ 매칭: matcher(돌린 드론 영상, 블록)\n④ 검증: MAGSAC++ inlier\n⑤ 위치·heading: 호모그래피 → pose_from_homography\n→ inlier가 가장 많은 각도의 결과를 고른다.\n   stop_inl > 0이면 각도를 차례로 해 보다가 inlier가 stop_inl 이상인 각도가 나오면 거기서 멈춘다 (조기 종료, 빠르게).\n"""\nimport time\n\nimport cv2\nimport numpy as np\n\nfrom .geometry import pose_from_homography, rot, turn\n\n\ndef localize(matcher, uav, block, gt, scale=1.0, angles=(0.0,), px_tol=8.0, stop_inl=0):\n    """각도별 기록 목록과 고른 결과(best, 없으면 None)를 돌려준다.\n\n    gt = (x_norm, y_norm, theta): 위치 추정에는 쓰지 않고, 대응점이 맞는지(good_*) 채점하는 데만 쓴다.\n    """\n    hu, wu = uav.shape[:2]; cu = np.array([wu / 2, hu / 2])\n    U = cv2.resize(uav, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC) if scale != 1 else uav\n    B = cv2.resize(block, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC) if scale != 1 else block\n    xn, yn, th = gt; C = np.array([256 + xn * 128, 256 + yn * 128]); Rg = rot(th)\n    recs = []\n    for a in angles:\n        t0 = time.time()\n        p0, p1, H, inl = matcher(turn(U, a), B)\n        dt = time.time() - t0\n        p0 = np.asarray(p0, np.float64) / scale; p1 = np.asarray(p1, np.float64) / scale\n        # 원래 드론 영상 좌표로 되돌림: p_turned = cu + R(a) (p - cu)  ->  p = cu + R(a)^T (p_turned - cu)\n        po = (p0 - cu) @ rot(a) + cu if len(p0) else p0\n        good = np.linalg.norm(C + (po - cu) @ Rg.T - p1, axis=1) < px_tol if len(p0) else np.zeros(0, bool)\n        r = dict(angle=a, n_match=int(len(p0)), n_inl=int(inl.sum()), sec=dt,\n                 good_match=float(good.mean()) if len(good) else np.nan,\n                 good_inl=float(good[inl].mean()) if inl.sum() else np.nan)\n        if H is not None and inl.sum() >= 4:\n            r.update(pose_from_homography(H, scale, a, cu))\n        recs.append(r)\n        if stop_inl > 0 and \'px\' in r and r[\'n_inl\'] >= stop_inl:\n            break\n    ok = [r for r in recs if \'px\' in r]\n    best = max(ok, key=lambda r: r[\'n_inl\']) if ok else None\n    return recs, best\n', 'imatch/__init__.py': '"""Trone 이미지 매칭 파이프라인: 드론 영상 + 지도 블록 → 위치·heading (학습 없음).\n\n  geometry.py  ④⑤ MAGSAC++ 검증, 호모그래피 → 위치·heading\n  matchers.py  ②③ 특징 추출·매칭 (LoMa, RoMa v2, SIFT ...)\n  pipeline.py  ①~⑤ 한 샘플 처리 (회전 탐색 포함)\n  blocks.py    ⑥ 후보 블록 자르기, 이웃 블록 좌표\n  nav.py       1 km 닫힌 루프 항법 (2D 크롭, 3D 렌더 입력)\n  metrics.py   팀 공통 평가 규칙\n"""\ndef __getattr__(name):\n    # CPU evaluation should not require GPU/image-processing dependencies.\n    if name == \'get_matcher\':\n        from .matchers import get_matcher\n        return get_matcher\n    if name == \'localize\':\n        from .pipeline import localize\n        return localize\n    raise AttributeError(name)\n', 'map_cache.py': 'class StepMapMatcher:\n    """Instrument both variants. Reuse only the second image of the pinned LoMa pair API."""\n    def __init__(self, matcher, reuse):\n        self.matcher = matcher\n        self.reuse = reuse\n        self.block = self.features = None\n        self.hits = self.extractions = 0\n        model = matcher.m.matcher\n        self.extract = model.detect_and_describe\n        model.detect_and_describe = self.describe\n\n    def __call__(self, uav, block):\n        if block is not self.block:\n            self.block = block\n            self.features = None\n        self.image_index = 0\n        result = self.matcher(uav, block)\n        if self.image_index != 2:\n            raise RuntimeError(\'Expected UAV and map extraction once per LoMa pair\')\n        return result\n\n    def describe(self, image, count):\n        self.image_index += 1\n        if self.image_index == 1:\n            return self.extract(image, count)\n        if self.image_index != 2:\n            raise RuntimeError(\'Unexpected feature extraction order\')\n        if self.reuse and self.features is not None:\n            self.hits += 1\n            return self.features\n        self.extractions += 1\n        result = self.extract(image, count)\n        if self.reuse:\n            self.features = (result[0], result[1], None, None)\n        return result\n', 'alto.py': 'import sys,json,time\nfrom pathlib import Path\nimport torch,numpy as np,pandas as pd,cv2\nimport random\nfrom imatch.matchers import get_matcher\nfrom imatch.pipeline import localize\nfrom map_cache import StepMapMatcher\n\nrun,data,maps=map(Path,sys.argv[1:4])\nmode=sys.argv[4]\nframes=pd.read_csv(run/\'frames.csv\')\ntiles={tuple(t) for t in json.loads((run/\'grid.json\').read_text())}\ndef rgb(p):\n    return cv2.cvtColor(cv2.imread(str(p)),cv2.COLOR_BGR2RGB)\ndef tile_path(key): return maps/f\'{key[0]}_{key[1]}.jpg\'\ndef locate_r1_r4(guess, streak, available, match, recover):\n    center = tuple(np.rint(guess / 256).astype(int))\n    attempts = 0\n    def probe(key, threshold, inside):\n        nonlocal attempts\n        if key not in available:\n            return None\n        records, best = match(key)\n        attempts += len(records)\n        if best is None or best[\'n_inl\'] < threshold:\n            return None\n        offset = np.array([best[\'px\'], -best[\'py\']])\n        if inside and np.abs(offset).max() > 1.1:\n            return None\n        return np.array(key)*256 + offset*128\n\n    position = probe(center, 15, False)\n    if position is not None:\n        return position, 0, attempts, \'r1\', False\n    streak += 1\n    if not recover:\n        return guess, streak, attempts, \'guess\', False\n    wide = streak >= 5\n    candidates = []\n    for dx in (-1, 0, 1):\n        for dy in (1, 0, -1):  # northing increases opposite to map row\n            if (dx,dy)==(0,0) or (not wide and abs(dx)+abs(dy)>1):\n                continue\n            position = probe((center[0]+dx, center[1]+dy), 20 if wide else 15, True)\n            if position is not None:\n                candidates.append(position)\n    if candidates:\n        return min(candidates,key=lambda p:np.linalg.norm(p-guess)), 0, attempts, \'r4\', wide\n    return guess, streak, attempts, \'guess\', wide\n\nassert mode in (\'r1\',\'r1_r4\')\nrandom.seed(2026); np.random.seed(2026); torch.manual_seed(2026); cv2.setRNGSeed(2026)\nbase=get_matcher(\'loma\',\'cuda\')\nwith torch.inference_mode():\n    base(cv2.resize(rgb(data/Path(frames.name.iloc[0]).name),(256,256)),rgb(tile_path(sorted(tiles)[0])))\n    torch.cuda.synchronize()\n    matcher=StepMapMatcher(base,True)\n    short=[];start=0\n    for j in range(1,len(frames)):\n        if frames.distance.iloc[j]-frames.distance.iloc[start]>=800:\n            short.append((start,j+1)); start=j\n    assert short\n    summaries=[]\n    for protocol,parts in [(\'short\',short),(\'continuous\',[(0,len(frames))])]:\n        rows=[]; torch.cuda.synchronize();t=time.perf_counter()\n        for seg,(lo,hi) in enumerate(parts):\n            rng=np.random.default_rng(2026+seg);bias=rng.normal(0,3)\n            truth=frames[[\'easting\',\'northing\']].iloc[lo:hi].to_numpy()\n            est=truth[0].copy()\n            streak=0\n            for i in range(1,len(truth)):\n                ang=np.deg2rad(bias+rng.normal(0,2));c,s=np.cos(ang),np.sin(ang)\n                est+=np.array([[c,-s],[s,c]])@(truth[i]-truth[i-1])*(1+rng.normal(0,.03))\n                key=tuple(np.rint(est/256).astype(int))\n                q=cv2.resize(rgb(data/Path(frames.name.iloc[lo+i]).name),(256,256),interpolation=cv2.INTER_AREA)\n                def match(tile):\n                    return localize(matcher,q,rgb(tile_path(tile)),(0,0,0),angles=[0,90,180,270],stop_inl=15)\n                est,streak,cnt,used,wide=locate_r1_r4(est,streak,tiles,match,recover=(mode=="r1_r4"))\n                accepted=used!=\'guess\'\n                rows.append(dict(segment=seg,frame=int(frames.frame.iloc[lo+i]),x=est[0],y=est[1],n_ang=cnt,accepted=accepted,used=used,wide=wide,covered=key in tiles,error_m=np.linalg.norm(est-truth[i])))\n        torch.cuda.synchronize(); elapsed=time.perf_counter()-t\n        records=pd.DataFrame(rows); records.to_csv(run/f\'{protocol}_{mode}_steps.csv\',index=False)\n        finals=records.groupby(\'segment\').tail(1).error_m\n        summaries.append(dict(r4_recoveries=int((records.used==\'r4\').sum()),wide_steps=int(records.wide.sum()),mode=mode,protocol=protocol,segments=len(parts),steps=len(records),wall_s=elapsed,r4_calls=int((records.used!="r1").sum()) if mode=="r1_r4" else 0,median_m=records.error_m.median(),final_m=finals.median(),map_coverage=records.covered.mean()))\n    pd.DataFrame(summaries).to_csv(run/f\'{mode}_summary.csv\',index=False)\n'}
for name,source in sources.items():
    p=SOURCE/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_text(source,encoding='utf8')
def evaluate(mode):
    subprocess.run([str(PY),'-u',str(SOURCE/'alto.py'),str(RUN),str(IMAGES),str(maps),mode],check=True)

In [ ]:
evaluate('r1')

In [ ]:
evaluate('r1_r4')

In [ ]:
results=pd.concat([pd.read_csv(RUN/f'{m}_summary.csv') for m in ('r1','r1_r4')],ignore_index=True)
for protocol in results.protocol.unique():
    mask=results.protocol==protocol
    ref=results.loc[mask & (results['mode']=='r1')].iloc[0]
    a,b=[pd.read_csv(RUN/f'{protocol}_{m}_steps.csv').set_index(['segment','frame'],verify_integrity=True) for m in ('r1','r1_r4')]
    assert a.index.equals(b.index)
    results.loc[mask,'delta_time_s']=results.loc[mask,'wall_s']-ref.wall_s
    results.loc[mask,'delta_final_m']=results.loc[mask,'final_m']-ref.final_m
results.to_csv(RUN/'comparison.csv',index=False)
display(results[['protocol','mode','steps','wall_s','delta_time_s','median_m','final_m','delta_final_m','r4_calls','r4_recoveries','wide_steps']])
archive=shutil.make_archive(str(RUN)+'_results','zip',RUN)
display(FileLink(str(Path(archive).relative_to('/kaggle/working'))))
